# AI Agent for the Titanic Dataset

We will be using the LangChain Agent framework to create an AI agent that can answer questions about the Titanic dataset.
The agent will be able to understand natural language queries and provide insights based on the data.
<br/><br/>
### Use of The LLM
<br/>
The AI agent needs an LLM to be able to perform its work. We can use any LLM that has tooling support.
Of course, the quality of the output may vary based on the model selection, however, this doesn't impact the way we create an Agent

<br/>
<br/>

In here we'll br running lasters ___gpt-oss___ (at the time of creation of this notebook) model over ollama, which doesn't require and API key and connection to an external interface.

### Word of Caution
This is just an example program and is not intented to be used in production without creating adequate security related safeguards. Any AI agent or LLM with tooling support can access, read and modify the data.
<br/><br/>
___use this notebook only for understanding of an AI agent and how to create one with LangChain___
<br/><br/>

In [ ]:
import pandas as pd
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain.agents import create_agent


## System Prompt and Local (user) prompts
<br/>
Though not mandatory, but always a good practice to create a system prompt to set the context, area, and boundaries (permission) for the AI agents. Doing this will not help the LLM but will also allow AI agents to be more specific to the task at hand rather than thinking about all system level parameters and permissions.

<br/><br/>
Remeber, that an agent and LLM may interfact multiple times to give you the requested answer, so its important to create the system prompt in advance
<br/><br/>


In [ ]:
SYSTEM_PROMPT = """You are an expert Python Pandas programmer and analyst. You answer questions
related to the provided dataset by running pandas code against a DataFrame called `df`.

IMPORTANT — follow these rules exactly:
1. To get data you MUST call the `execute_pandas_code` tool with valid Python pandas code.
2. The code you pass must be a one or more Python expression that operates on `df`.
3. Do NOT import anything — pandas is already available as `pd` and the DataFrame as `df`.
4. Do NOT use print(); just write the expression and its result will be returned.
5. If the tool returns an error, revise the code and try again (up to 3 times).
6. Use vectorized operations; avoid iterating over rows.
7. After receiving the result, summarise the answer in plain language.

Here are the columns in the DataFrame:
{columns}

Here are the first 5 rows:
{head}
"""

<br/><br/>

In [ ]:
# create the dataframe
df = pd.read_csv("Titanic-Dataset.csv")

<br/><br/>

___Provide the required data to the SYSTEM_PROMPT___

<br/><br/>

In [ ]:
SYSTEM_PROMPT = SYSTEM_PROMPT.format(
    columns=df.columns.tolist(),
    head=df.head().to_string()
)

<br/><br/>

___Here is the first 5 rows of the dataset___
<br/>

In [ ]:
df.head()

<br/><br/>

### The Tool
<br/>
AI agents need tools to be able to perform their work. In terms of a python program, the simplest tool can be a function that will be called by the agent.
<br/>
In the real production environment, there will be multiple tools that will be used by the agent, and they can be of different types (APIs, databases, python functions, etc..).

<br/><br/>
_Let's create a simple function which will execute all pandas code passed to it and will be written by the LLM with the help of AI Agent_

<br/><br/>


In [ ]:
@tool
def execute_pandas_code(code: str) -> str:
    '''tool be called by the agent to execute pandas code'''
    df_agent = df
    try:
        result = eval(code, {"__builtins__": {}}, {"df": df_agent, "pd": pd})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

<br/><br/>

___Now we have our tool ready, let's create the instance of LLM which will be used by our agent. We will be using Ollama here, but any LLM with tooling support can be used here___

<br/><br/>

In [ ]:
llm = ChatOllama(model="gpt-oss:latest", temperature=0.0)

<br/><br/>

In [ ]:
agent_tools = [
            execute_pandas_code,
        ]

#create agent using Langchain
agent = create_agent(
            model=llm,
            tools=agent_tools,
            system_prompt=SYSTEM_PROMPT
        )

<br/><br/>

___Now its time to trigger our agent with a question related to titanic dataset___

<br/><br/>

In [ ]:
question = "How many passengers survived in the titanic disaster"

<br/><br/>

In [ ]:
# invoke the Agent

result = agent.invoke(
            {"messages": question}
        )

<br/><br/>

In [ ]:
result["messages"][-1].content

<br/><br/>
<br/><br/>

### Streaming the answer from the agent
The agent can also stream the answer back to us and can show the conversation.

In [ ]:
for next_step in agent.stream(
            {"messages": question},
            stream_mode="values",
    ):
        next_step["messages"][-1].pretty_print()